# Hardread TCG — BC + Dual-PPO Pipeline

Phases:
0. Setup (attach episode datasets + cg-lib engine, install deps)
1. Replay parsing — discover episode JSON schema, extract `(obs, action)` pairs, Elo filter
2. Featurizer — `featurize(obs_dict) -> (state_vec, option_matrix, legal_mask)` (pure numpy)
3. Behavioural cloning — JAX/flax MLP, score-per-option head, init weights for PPO
4. Dual-PPO self-play — two agents, rolling checkpoint pool, Elo health signal
5. Eval vs 3 kiyotah rule-based agents + package `submission.tar.gz`

**Storage policy:** all heavy data (21 GB episode datasets, BC pairs, PPO rollouts, checkpoints) stays in `/kaggle/working/`. Local repo only holds source code.

In [ ]:
# ── Phase control flags ──
# Set to False to skip phases. Only enable what you need to run.
RUN_PHASE_1 = True   # replay parsing (always needed for BC data)
RUN_PHASE_3 = True   # BC training
RUN_PHASE_4 = False  # PPO self-play ── DISABLE if only testing BC
RUN_PHASE_5 = False  # eval + submit ── DISABLE if training only
print(f"Phases: P1={RUN_PHASE_1} P3={RUN_PHASE_3} P4={RUN_PHASE_4} P5={RUN_PHASE_5}")


## Phase 0 — Setup

Attach these Kaggle datasets to the notebook (right sidebar → Add Input):
- `kaggle/pokemon-tcg-ai-battle-episodes-2026-06-17` (top avg 1259)
- `kaggle/pokemon-tcg-ai-battle-episodes-2026-06-18` (top avg 1327)
- `kaggle/pokemon-tcg-ai-battle-episodes-index` (manifest)
- Competition data tab (`pokemon-tcg-ai-battle`) — card data + cg-lib engine + deck lists

Enable GPU + Internet in the notebook settings.

In [ ]:
# Phase 0.1 — install + sanity check
import sys, os, glob, json, time

!pip -q install kaggle-environments==1.30.1 2>&1 | tail -3

try:
    import kaggle_environments
    print("kaggle_environments:", kaggle_environments.__version__ if hasattr(kaggle_environments, '__version__') else 'ok')
except Exception as e:
    print("kaggle_environments import failed:", e)

try:
    import jax, flax, optax
    print("jax", jax.__version__, "flax", flax.__version__, "optax", optax.__version__, "devices", jax.devices())
except Exception as e:
    print("jax stack missing:", e)

try:
    import cg.api as cabt_api
    print("cg.api available locally (lookahead/search API usable)")
except Exception as e:
    print("cg.api NOT importable here (expected on Kaggle host w/o vendored cg/):", str(e)[:80])

In [ ]:
# Phase 0.2 — enumerate attached inputs (datasets + competition data)
INPUT = "/kaggle/input"
print("Attached /kaggle/input tree (depth 2):")
for root, dirs, files in os.walk(INPUT):
    depth = root.replace(INPUT, '').count(os.sep)
    if depth > 2:
        dirs.clear()
        continue
    indent = '  ' * (depth + 1)
    print(f"{indent}{os.path.basename(root)}/" if depth > 0 else f"{INPUT}/")
    if depth >= 2:
        for f in sorted(files)[:10]:
            sz = os.path.getsize(os.path.join(root, f))
            print(f"{indent}  {sz:>10}  {f}")
        if len(files) > 10:
            print(f"{indent}  ... ({len(files)-10} more files)")

# Find the hardread-tcg-src dataset and add to sys.path
SRC_PATH = None
for root, dirs, files in os.walk(INPUT):
    if 'featurizer.py' in files and 'replay.py' in files:
        SRC_PATH = root
        break
    # check for src.zip (Kaggle zips folders)
    for d in dirs:
        candidate = os.path.join(root, d)
        if os.path.exists(os.path.join(candidate, 'featurizer.py')):
            SRC_PATH = candidate
            break
if SRC_PATH:
    sys.path.insert(0, os.path.dirname(SRC_PATH))  # parent so 'import src.X' works
    sys.path.insert(0, SRC_PATH)  # also add directly for flat imports
    print(f"\nsrc/ found at: {SRC_PATH}")
else:
    # Try unzipping src.zip if present
    import subprocess, glob
    zips = glob.glob(os.path.join(INPUT, '**/*src*.zip'), recursive=True)
    for z in zips:
        subprocess.run(['unzip', '-o', z, '-d', '/kaggle/working/'], capture_output=True)
    if os.path.exists('/kaggle/working/src/featurizer.py'):
        sys.path.insert(0, '/kaggle/working')
        SRC_PATH = '/kaggle/working/src'
        print(f'\nsrc/ unzipped to /kaggle/working/src/')
    else:
        print('\nWARNING: src/ not found — src files need to be available')

# Find and add cg/ engine to path
CG_PATH = None
for root, dirs, files in os.walk(INPUT):
    if 'game.py' in files and 'sim.py' in files and 'api.py' in files:
        CG_PATH = root
        break
if CG_PATH:
    sys.path.insert(0, os.path.dirname(CG_PATH))  # parent so 'import cg.game' works
    print(f'cg/ found at: {CG_PATH}')
    try:
        from cg.game import battle_start, battle_select, battle_finish
        print('cg.game import OK — PPO self-play can use cg engine directly')
    except Exception as e:
        print(f'cg.game import failed: {e}')
else:
    # Try unzipping cg.zip
    import subprocess, glob
    zips = glob.glob(os.path.join(INPUT, '**/*cg*.zip'), recursive=True)
    for z in zips:
        subprocess.run(['unzip', '-o', z, '-d', '/kaggle/working/'], capture_output=True)
    if os.path.exists('/kaggle/working/cg/game.py'):
        sys.path.insert(0, '/kaggle/working')
        CG_PATH = '/kaggle/working/cg'
        print(f'cg/ unzipped to /kaggle/working/cg/')
        try:
            from cg.game import battle_start, battle_select, battle_finish
            print('cg.game import OK')
        except Exception as e:
            print(f'cg.game import failed: {e}')
    else:
        print('cg/ not found — PPO self-play will use kaggle_environments instead')

# Copy deck.csv to /kaggle/working/ if found
import shutil
for root, dirs, files in os.walk(INPUT):
    if 'deck.csv' in files:
        shutil.copy2(os.path.join(root, 'deck.csv'), '/kaggle/working/deck.csv')
        print(f'deck.csv copied from {root}')
        break

## Phase 1 — Replay schema discovery

The cabt docs describe the *runtime* `obs_dict` but NOT the on-disk replay JSON format. This cell inspects one episode file to learn the schema before writing `src/replay.py`. We read only ONE small file to avoid loading 21 GB.

In [ ]:
# Phase 1.1 — locate episode JSON files across ALL of /kaggle/input/
# Datasets mount under /kaggle/input/datasets/<slug>/ — search recursively
EPISODE_DIRS = []
for root, dirs, files in os.walk(INPUT):
    # An episode dir is one that contains .json or .jsonl files
    if any(f.endswith((".json", ".jsonl")) for f in files):
        EPISODE_DIRS.append(root)
    if len(EPISODE_DIRS) > 50:
        break
print(f"Dirs with JSON files: {len(EPISODE_DIRS)}")
for d in EPISODE_DIRS[:10]:
    print(f"  {d}")

def list_top(d, n=15):
    out = []
    for root, dirs, files in os.walk(d):
        for f in files:
            out.append(os.path.join(root, f))
            if len(out) >= n:
                return out
    return out

if EPISODE_DIRS:
    d = EPISODE_DIRS[0]
    sample = list_top(d, 20)
    print(f"\n{d}  ({len(sample)} sample paths shown)")
    for p in sample[:20]:
        try:
            sz = os.path.getsize(p)
        except OSError:
            sz = -1
        print(f"  {sz:>10}  {os.path.relpath(p, d)}")
else:
    print("\nNo episode dirs found — check dataset attachment")

In [ ]:
# Phase 1.2 — inspect ONE episode file's top-level structure (pick smallest .json)
# Search ALL of /kaggle/input/ for episode JSON files
candidates = []
for root, dirs, files in os.walk(INPUT):
    for f in files:
        if f.endswith(".json") or f.endswith(".jsonl"):
            p = os.path.join(root, f)
            try:
                candidates.append((os.path.getsize(p), p))
            except OSError:
                pass
    if len(candidates) > 500:
        break
candidates.sort()
print(f"Total episode files found: {len(candidates)}")
if candidates:
    print("Smallest 5:", [(s, os.path.basename(p)) for s, p in candidates[:5]])
    ONE = candidates[0][1]
    print(f"\nInspecting: {ONE} ({candidates[0][0]} bytes)")
    with open(ONE, "r") as fh:
        raw = fh.read()
    print("file bytes:", len(raw))
    # Try JSON first; fall back to JSONL (one obj per line)
    try:
        obj = json.loads(raw)
        fmt = "json"
    except json.JSONDecodeError:
        lines = [l for l in raw.splitlines() if l.strip()]
        obj = json.loads(lines[0])
        fmt = f"jsonl ({len(lines)} lines)"
    print("format:", fmt)
    print("top-level type:", type(obj).__name__)
    if isinstance(obj, dict):
        print("top-level keys:", list(obj.keys()))
        for k, v in obj.items():
            tname = type(v).__name__
            if isinstance(v, (list, tuple)):
                tname += f"[{len(v)}]"
                inner = v[0] if v else None
                inner_t = type(inner).__name__ if inner is not None else "None"
                print(f"  {k:20} {tname:14} first_elem_type={inner_t}")
            elif isinstance(v, dict):
                print(f"  {k:20} dict keys={list(v.keys())[:12]}")
            else:
                print(f"  {k:20} {tname:14} value={str(v)[:80]}")
else:
    print("No JSON files found anywhere under /kaggle/input/ — check dataset attachments")
    obj = None

In [ ]:
# Phase 1.3 — drill into the episode structure: find per-game metadata + a turn/step record
# Looking for: (a) Elo/score fields per player, (b) deck identity, (c) the obs/action pair shape
if 'obj' in globals() and isinstance(obj, dict):
    def show(obj, path="root", depth=0, maxd=4):
        if depth > maxd:
            return
        if isinstance(obj, dict):
            for k, v in obj.items():
                p = f"{path}.{k}"
                if isinstance(v, dict):
                    print(f"{'  '*depth}{k}: dict {len(v)}keys -> {list(v.keys())[:8]}")
                    show(v, p, depth+1, maxd)
                elif isinstance(v, list):
                    print(f"{'  '*depth}{k}: list[{len(v)}]", end="")
                    if v and isinstance(v[0], dict):
                        print(f" elem0.keys={list(v[0].keys())[:10]}")
                        if depth < maxd:
                            show(v[0], f"{p}[0]", depth+1, maxd)
                    elif v:
                        print(f" elem0={str(v[0])[:60]}")
                    else:
                        print()
                else:
                    print(f"{'  '*depth}{k}: {type(v).__name__} = {str(v)[:70]}")
        elif isinstance(obj, list):
            print(f"{'  '*depth}list[{len(obj)}] elem0:")
            if obj:
                show(obj[0], path+"[0]", depth+1, maxd)
    show(obj, maxd=4)
else:
    print("obj not available — cell 1.2 found no episode file to inspect")

In [ ]:
# Phase 1.4 — locate the 3 named deck lists (Lucario / Crustle / Alakazam) in competition data
# The cabt engine has no named-deck concept; these must be Kaggle competition assets.
DECK_CANDIDATES = []
for root, dirs, files in os.walk(INPUT):
    for f in files:
        low = f.lower()
        if (low.endswith(".csv") or low.endswith(".txt") or low.endswith(".tsv")) and (
            "deck" in low or "lucario" in low or "crustle" in low or "alakazam" in low
        ):
            DECK_CANDIDATES.append(os.path.join(root, f))
        elif any(name in low for name in ("lucario", "crustle", "alakazam")):
            DECK_CANDIDATES.append(os.path.join(root, f))
print("Deck-list candidates by name:")
for p in DECK_CANDIDATES:
    try:
        sz = os.path.getsize(p)
    except OSError:
        sz = -1
    print(f"  {sz:>8}  {p}")

# Fallback: any CSV that looks like a 60-card deck (lines parse as ints)
print("\nFallback — CSVs whose lines are mostly ints (potential 60-card decks):")
for root, dirs, files in os.walk(INPUT):
    for f in files:
        if not f.lower().endswith(".csv"):
            continue
        p = os.path.join(root, f)
        try:
            if os.path.getsize(p) > 200000:
                continue
            with open(p) as fh:
                lines = [l.strip() for l in fh if l.strip()]
            n_int = sum(1 for l in lines if l.split(",")[0].isdigit())
            if n_int >= 55:
                print(f"  {p}  ({n_int} int lines, first={lines[0][:40]})")
        except Exception:
            pass

In [ ]:
# Phase 1.5 — run a tiny cabt game to confirm the live obs_dict shape (ground truth for featurizer)
# cg is NOT importable on Kaggle host, so use the kaggle_environments cabt env.
from kaggle_environments import make

DEMO_DECK = [278, 278, 278, 278, 7, 7, 7, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
             3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
             3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]

def spy_agent(obs):
    # Record one real obs then pick random legal move
    import random
    if obs is None or obs.get("select") is None:
        if not hasattr(spy_agent, "_init_seen"):
            print("INIT obs:", json.dumps({k: type(v).__name__ for k, v in (obs or {}).items()}, indent=0))
            spy_agent._init_seen = True
        return []
    if not hasattr(spy_agent, "_seen"):
        print("\nLIVE obs_dict top keys:", list(obs.keys()))
        sel = obs["select"]
        print("select keys:", list(sel.keys()))
        print("select.type/context/minCount/maxCount:", sel.get("type"), sel.get("context"), sel.get("minCount"), sel.get("maxCount"))
        print("select.option len:", len(sel.get("option", [])), "option0:", json.dumps(sel["option"][0])[:200] if sel.get("option") else None)
        cur = obs.get("current")
        if cur:
            print("current keys:", list(cur.keys()))
            ps = cur.get("players", [])
            for i, pl in enumerate(ps):
                print(f"  player[{i}] keys:", list(pl.keys()))
                print(f"    active={pl.get('active')} handCount={pl.get('handCount')} deckCount={pl.get('deckCount')} bench={len(pl.get('bench',[]))}")
        spy_agent._seen = True
    sel = obs["select"]
    return random.sample(list(range(len(sel["option"]))), sel["maxCount"])

env = make("cabt", configuration={"decks": [DEMO_DECK, DEMO_DECK]})
env.run([spy_agent, spy_agent])
print("\nSpy game done.")

## Phase 1 (cont.) — Replay parsing

After schema discovery (cells above), walk episode JSONs, extract `(obs, action)` pairs,
filter by Elo > 1100, tag deck identity, save to `/kaggle/working/bc_pairs/*.npz`.

**TODO after first run:** Fill in the exact field names from the schema discovery output
(episode Elo field, deck identity field, per-turn obs/action record path).

In [ ]:
# Phase 1.6 — parse episodes into BC shards using src/replay.py
# PRIORITY 2: No max_files cap — use ALL episodes from both datasets.
# Expected: ~14k files -> ~250-350k pairs (we got ~22.5k from 1k files).
# If the count comes back near 22.5k, the shard-overwrite bug is NOT fixed.
import sys, os, time, glob
sys.path.insert(0, '/kaggle/working')
from src.replay import write_shards

# Clean any old shards first (ensures we don't inherit stale data)
OUT_DIR = '/kaggle/working/bc_pairs'
import shutil
if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

# Find episode dataset dirs (confirmed path pattern from Kaggle)
EPISODE_DIRS = []
for root, dirs, files in os.walk(INPUT):
    if any(f.endswith('.json') for f in files) and 'episodes' in root.lower():
        EPISODE_DIRS.append(root)
print(f'Episode dirs: {EPISODE_DIRS}')

total_pairs = 0
for d in EPISODE_DIRS:
    n = write_shards(
        episode_dir=d,
        out_dir=OUT_DIR,
        shard_size=5000,
        max_files=None,  # NO CAP — use all episodes
        winner_only=True,
        deck_id=0,
    )
    total_pairs += n

# Verify shard count — overwrite bug guard
shards = sorted(glob.glob(os.path.join(OUT_DIR, 'shard_*.npz')))
print(f'\nTotal BC pairs: {total_pairs} -> {OUT_DIR}')
print(f'Shards on disk: {len(shards)} (range: {os.path.basename(shards[0])} to {os.path.basename(shards[-1])})')
expected_min = 200000  # ~22.5k from 1k files -> expect at least 200k from ~14k files
if total_pairs < expected_min:
    print(f'WARNING: pair count {total_pairs} is below expected minimum {expected_min}.')
    print('This may indicate the shard-overwrite bug is NOT fixed.')
else:
    print(f'Pair count {total_pairs} is in expected range (>={expected_min}). Overwrite bug fix confirmed.')

## Phase 2 — Featurizer (in src/)

`src/featurizer.py` is pure-numpy and validated against real cabt obs.
Already imported via the hardread-tcg-src dataset (Phase 0.2 setup).

## Phase 3 — Behavioural Cloning

Train the Flax MLP on the BC shards. Score-per-option head with Plackett-Luce loss.
Saves weights to `/kaggle/working/weights/bc/` (orbax) and `/kaggle/working/weights/bc.npz` (numpy export for PPO init + submission).

In [ ]:
# Phase 3 — BC training (Priority 2: converged, properly-validated)
# 25 epochs with early stopping (patience=5), 90/10 train/val split,
# per-epoch val loss + val multi-option top-1 accuracy tracking.
import sys
sys.path.insert(0, "/kaggle/working")
import jax, numpy as np
from src.train_bc import BCConfig, train_bc
from src.policy_numpy import export_params

bc_cfg = BCConfig(
    lr=3e-4,
    batch_size=256,
    epochs=25,  # up from 3 — early stopping will cut it if val plateaus
    embed_dim=32,
    trunk_hidden=256,
    seed=0,
    shard_glob="/kaggle/working/bc_pairs/*.npz",
    out_dir="/kaggle/working/weights/bc",
    val_split=0.1,
    early_stopping_patience=5,
)
bc_state, bc_history = train_bc(bc_cfg)

# Export to numpy .npz for PPO init + submission packaging
export_params(bc_state.params, "/kaggle/working/weights/bc.npz")
print("BC weights exported to /kaggle/working/weights/bc.npz")
print(f"\nFinal val acc (multi-option): {bc_history[-1]['val_acc_multi']:.4f}")
print(f"Best epoch: {max(h['epoch'] for h in bc_history if h['is_best'])}")

## Phase 4 — Dual-PPO Self-Play

Two PPO agents (A, B) initialized from BC weights. Alternate frozen/active.
Opponent sampled from rolling checkpoint pool. Elo logged as health signal.
cg engine runs natively on Kaggle (no QEMU).

In [ ]:
if not RUN_PHASE_4:
    print('Phase 4 skipped (RUN_PHASE_4=False)')
else:
    # Phase 4 — Dual-PPO self-play
    import sys, json, os
    sys.path.insert(0, "/kaggle/working")
    import numpy as np, jax
    from src.ppo import PPOConfig, init_ppo_state
    from src.selfplay import SelfPlayConfig, run_selfplay
    from src.train_bc import load_shards, init_state as bc_init, BCConfig
    from src.policy import PolicyNet
    from src.policy_numpy import load_params, export_params
    from src.featurizer import featurize
    
    # Load BC data for model init shape
    bc_data = load_shards("/kaggle/working/bc_pairs/*.npz")
    sample = {k: v[:2] for k, v in bc_data.items()}
    
    ppo_cfg = PPOConfig(lr=3e-4, embed_dim=32, trunk_hidden=256, seed=0)
    
    # Init PPO states from BC weights (load .npz -> Flax params -> TrainState)
    bc_npz = "/kaggle/working/weights/bc.npz"
    if os.path.exists(bc_npz):
        # Load BC numpy weights and convert to Flax params tree
        bc_flat = np.load(bc_npz)
        bc_params = {}
        for key in bc_flat.files:
            parts = key.split("/")
            node = bc_params
            for p in parts[:-1]:
                node = node.setdefault(p, {})
            node[parts[-1]] = jax.numpy.asarray(bc_flat[key])
        bc_params = {"params": bc_params}
        import optax
        from flax.training import train_state
        model = PolicyNet(embed_dim=32, trunk_hidden=256)
        tx = optax.adamw(learning_rate=ppo_cfg.lr)
        state_a = train_state.TrainState.create(apply_fn=model.apply, params=bc_params, tx=tx)
        # Agent B: same BC init + small noise for diversity
        rng = jax.random.PRNGKey(1)
        bc_params_b = jax.tree_util.tree_map(lambda x: x + jax.random.normal(rng, x.shape) * 0.01, bc_params)
        state_b = train_state.TrainState.create(apply_fn=model.apply, params=bc_params_b, tx=tx)
        print(f"Loaded BC weights from {bc_npz} for PPO init")
    else:
        print("No BC weights found — using random init")
        state_a = init_ppo_state(ppo_cfg, sample)
        state_b = init_ppo_state(PPOConfig(lr=3e-4, embed_dim=32, trunk_hidden=256, seed=1), sample)
    
    # Load the 3 distinct decks (Priority 1 fix — no more mirror-match bug)
    from src.selfplay import load_decks, verify_decks_distinct
    DECK_DIR = None
    for root, dirs, files in os.walk(INPUT):
        if 'deck_lucario.csv' in files:
            DECK_DIR = root
            break
    if DECK_DIR is None:
        DECK_DIR = '/kaggle/working'
    DECKS = load_decks(DECK_DIR, ["Lucario", "Crustle", "Alakazam"])
    verify_decks_distinct(DECKS)
    
    # Engine wrapper — cg.game is on the path (added in Phase 0.2)
    class CgEngine:
        def battle_start(self, d0, d1):
            from cg.game import battle_start
            return battle_start(d0, d1)
        def battle_select(self, action):
            from cg.game import battle_select
            return battle_select(action)
        def battle_finish(self):
            from cg.game import battle_finish
            battle_finish()
    
    # Test that cg works
    try:
        from cg.game import battle_start, battle_finish
        obs, _ = battle_start(DECKS["Lucario"], DECKS["Lucario"])
        battle_finish()
        print(f"cg engine test OK: obs is None? {obs is None}")
        engine = CgEngine()
    except Exception as e:
        print(f"cg engine not available: {e}")
        print("Falling back to kaggle_environments for rollouts (slower)")
        engine = None  # selfplay will need a different engine wrapper
    
    if engine is not None:
        sp_cfg = SelfPlayConfig(
            episodes_per_round=200,  # more rollouts per round for stable gradients
            rounds=15,
            deck_lists=DECKS,
            ppo=ppo_cfg,  # tuned: lr=1e-4, clip=0.3, entropy=0.02, epochs=8
            temperature=0.5,  # less random than 1.0; still explores
            per_move_budget_s=0.2,
            pool_size=4,
        )
        summary = run_selfplay(engine, state_a, state_b, sp_cfg)
        print(f"Final Elo: {summary['final_elo']}")
    
        # Save final weights
        best_state = summary['state_a'] if summary['final_elo']['A'] > summary['final_elo']['B'] else summary['state_b']
        export_params(best_state.params, "/kaggle/working/weights/ppo_final.npz")
        print("Final PPO weights saved to /kaggle/working/weights/ppo_final.npz")
    else:
        print("Skipping PPO self-play (no engine)")

## Phase 5 — Eval + Submit

1. Eval vs 3 kiyotah rule-based agents in the cabt env
2. Package `submission.tar.gz` (main.py + deck.csv + weights.npz + featurizer.py + policy_numpy.py)
3. Verify never-crash + per-move time limit
4. Submit to Kaggle competition

In [ ]:
if not RUN_PHASE_5:
    print('Phase 5 skipped (RUN_PHASE_5=False)')
else:
    # Phase 5.1 — Eval vs kiyotah rule-based agents
    # The 3 kiyotah agents are Kaggle notebooks — import their agent functions here.
    # If not available, eval vs random/bandit as a smoke test.
    import sys, json, time
    sys.path.insert(0, "/kaggle/working")
    import numpy as np
    from kaggle_environments import make
    
    # Load the neural agent (standalone, like in the submission tarball)
    from src.policy_numpy import load_params, numpy_forward, numpy_select
    from src.featurizer import featurize
    
    PARAMS = load_params("/kaggle/working/weights/ppo_final.npz")
    
    def neural_agent(obs_dict):
        if obs_dict is None or obs_dict.get("select") is None:
            with open("/kaggle/working/deck.csv") as f:
                return [int(l.strip()) for l in f if l.strip()][:60]
        sel = obs_dict["select"]
        opts = sel.get("option") or []
        mc = int(sel.get("maxCount", 1))
        if not opts or mc <= 0: return []
        try:
            feats = featurize(obs_dict, deck_id=0)
            logits = numpy_forward(PARAMS, feats)
            chosen = numpy_select(logits, max_count=mc, temperature=0.0)
            if len(chosen) > 0: return [int(i) for i in chosen]
        except Exception: pass
        import random
        return random.sample(range(len(opts)), min(mc, len(opts)))
    
    def random_agent(obs_dict):
        import random
        if obs_dict is None or obs_dict.get("select") is None:
            with open("/kaggle/working/deck.csv") as f:
                return [int(l.strip()) for l in f if l.strip()][:60]
        sel = obs_dict["select"]
        return random.sample(range(len(sel["option"])), sel["maxCount"])
    
    # Eval: neural vs random (smoke test — replace with kiyotah agents when available)
    with open("/kaggle/working/deck.csv") as f:
        deck = [int(l.strip()) for l in f if l.strip()][:60]
    
    N_EVAL = 20
    env = make("cabt", configuration={"decks": [deck, deck]})
    wins = 0
    for i in range(N_EVAL):
        t0 = time.time()
        traj = env.run([neural_agent, random_agent])
        # extract winner
        last = traj[-1] if traj else []
        winner = -1
        if isinstance(last, list) and len(last) >= 2:
            r0 = last[0].get("reward") if isinstance(last[0], dict) else None
            r1 = last[1].get("reward") if isinstance(last[1], dict) else None
            if r0 is not None and r1 is not None:
                winner = 0 if r0 > r1 else (1 if r1 > r0 else -1)
        if winner == 0: wins += 1
        print(f"Game {i+1}: winner=P{winner} ({time.time()-t0:.1f}s)")
    print(f"\nNeural vs Random: {wins}/{N_EVAL} wins ({wins/N_EVAL:.1%})")

In [ ]:
if not RUN_PHASE_5:
    print('Phase 5 skipped (RUN_PHASE_5=False)')
else:
    # Phase 5.2 — Package submission.tar.gz
    import sys, os, shutil
    sys.path.insert(0, "/kaggle/working")
    from src.submit import package_submission
    
    # Find main_neural.py in the dataset (configs.zip was auto-extracted)
    agent_path = None
    for root, dirs, files in os.walk(INPUT):
        if 'main_neural.py' in files:
            agent_path = os.path.join(root, 'main_neural.py')
            break
    if agent_path is None:
        # Fallback: copy from /kaggle/working if unzipped there
        for p in ['/kaggle/working/configs/main_neural.py', '/kaggle/working/main_neural.py']:
            if os.path.exists(p):
                agent_path = p
                break
    if agent_path is None:
        print('ERROR: main_neural.py not found — copying from dataset')
        # Last resort: find it anywhere and copy
        import subprocess, glob
        zips = glob.glob(os.path.join(INPUT, '**/*configs*.zip'), recursive=True)
        for z in zips:
            subprocess.run(['unzip', '-o', z, '-d', '/kaggle/working/'], capture_output=True)
        agent_path = '/kaggle/working/configs/main_neural.py'
    print(f'Agent path: {agent_path}')
    
    # Find featurizer.py and policy_numpy.py
    feat_path = os.path.join(SRC_PATH, 'featurizer.py') if SRC_PATH else None
    pnumpy_path = os.path.join(SRC_PATH, 'policy_numpy.py') if SRC_PATH else None
    if not feat_path or not os.path.exists(feat_path):
        feat_path = '/kaggle/working/src/featurizer.py'
    if not pnumpy_path or not os.path.exists(pnumpy_path):
        pnumpy_path = '/kaggle/working/src/policy_numpy.py'
    
    tar_path = package_submission(
        output_dir="/kaggle/working",
        version="ppo_v1",
        deck_path="/kaggle/working/deck.csv",
        agent_path=agent_path,
        featurizer_path=feat_path,
        policy_numpy_path=pnumpy_path,
        weights_path="/kaggle/working/weights/ppo_final.npz",
    )
    print(f"\nSubmission ready: {tar_path}")
    print(f"Size: {os.path.getsize(tar_path)} bytes")
    print(f"\nTo submit to the competition:")
    print(f"  kaggle competitions submit pokemon-tcg-ai-battle -f {tar_path} -m 'PPO v1: BC+PPO neural agent'"